## Select residues within certain distance from the substrate in pymol

In [1]:
''' find residues within certain distance from the substrate'''
select near_residues_20A, byres (protein and name CA within 20 of (substrate and name P))

# select protein and ligand and name them as protein and substrate 
# protein and name CA: selects only the alpha-carbon atoms from the protein
# substrate and name P: selects only the phosphate atoms from the substrate (nucleic acids)
# within 18 of Finds atom within 18 A of each other 
# byres: ensures that entire resiudes are selected if any of their atoms meet the criteria 

SyntaxError: invalid syntax (4136223627.py, line 2)

In [2]:
''' save residues position indices in a text file '''
# Step 1: Initialize an empty list
residue_indices_20A = []

# Step 2: Collect residue indices from the selection
cmd.iterate("near_residues_20A", "residue_indices_20A.append(resi)")

# Step 3: Remove duplicates and sort the list
residue_indices_20A = sorted(set(residue_indices_20A), key=lambda x: int(x))

# Step 4: Define the file path
file_path = '/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6a/PE6a_AF3_20A_residue_indices.txt'

# Step 5: Save the list to a text file
with open(file_path, 'w') as outfile: outfile.write(', '.join(residue_indices_20A))

NameError: name 'cmd' is not defined

## Convert the active site residue indices in the substrate-bound homolog to those in target protein

In [2]:
from Bio import AlignIO

In [3]:
'''import the residues text file generated from pymol'''
# Step 1: Define the file path
file_path = '/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/18A_residue_indices.txt'

# Step 2: Read the contents of the file
with open(file_path, 'r') as infile:
    data = infile.read()

# Step 3: Convert the string data to a list
residue_indices = data.split(', ')

# Step 4: (Optional) Convert residue indices to integers
residue_indices = [int(res) for res in residue_indices]

FileNotFoundError: [Errno 2] No such file or directory: '/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/18A_residue_indices.txt'

In [5]:
# MMLV_deltaRNaseH_substrate_bound has some wired index annotations
residue_indices_adjusted =[]
for r in residue_indices:
    if r <449:
        r2=r-23
    if r>449:
        r2 = r-23-6
    residue_indices_adjusted.append(r2)

In [6]:
'''if use a homolog structure, the codes below can convert the homolog residue positions to your target protein positions'''
alignment = AlignIO.read('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/Alignment_MMLV_deltaRNaseH_PE6d.fasta', "fasta")

# define the function to map the homolog residues to the target protein residues by sequence alignment
def index_convert(resList):

  outList = []
  outList_num = []
  res_counter_A = 0
  res_counter_B = 0
  for pos in range(alignment.get_alignment_length()):

    if alignment[1][pos] != '-':
      res_counter_B += 1

    if alignment[0][pos] != '-':
      res_counter_A += 1
      if res_counter_A in resList:
        outList.append(alignment[1][res_counter_B])
        outList_num.append(res_counter_B)
    
  return(outList,outList_num)

In [7]:
residues, num = index_convert(residue_indices_adjusted)
PDB_fix_Res = num
file_path = '/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PDB_fix_Res.txt'
with open(file_path, 'w') as outfile:
    for item in PDB_fix_Res:
        outfile.write(str(item) + '\n')

## Compare fixed residues between AF3 and experimental homolog structure 

In [21]:
import re

# function to read the text file 
def read_file(file_path):
    with open(file_path, 'r') as infile:
        data = infile.read()
        # Split on commas and newlines, strip whitespace, and filter out empty strings
        data_list = [item.strip() for item in re.split(r',\s*|\n', data) if item.strip()]
        # Convert the list items to integers
        data_list = [int(item) for item in data_list]
    return data_list

# read the text file
PE6a_AF3_15A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6a/PE6a_AF3_15A_residue_indices.txt')
PE6a_substrate_bound_AF3_15A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6a/PE6a_substrate_bound_AF3_15A_residue_indices.txt')
PE6a_AF3_18A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6a/PE6a_AF3_18A_residue_indices.txt')
PE6a_substrate_bound_AF3_18A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6a/PE6a_substrate_bound_AF3_18A_residue_indices.txt')
PE6a_AF3_20A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6a/PE6a_AF3_20A_residue_indices.txt')
PE6a_substrate_bound_AF3_20A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6a/PE6a_substrate_bound_AF3_20A_residue_indices.txt')

In [22]:
# Convert the lists to sets for efficient comparison
PE6a_AF3_15A_set = set(PE6a_AF3_15A)
PE6a_substrate_bound_AF3_15A_set = set(PE6a_substrate_bound_AF3_15A)
PE6a_AF3_18A_set = set(PE6a_AF3_18A)
PE6a_substrate_bound_AF3_18A_set = set(PE6a_substrate_bound_AF3_18A)
PE6a_AF3_20A_set = set(PE6a_AF3_20A)
PE6a_substrate_bound_AF3_20A_set = set(PE6a_substrate_bound_AF3_20A)

def compare_indices(set1, set2, set1_name='Set 1', set2_name='Set 2'):
    """
    Compares two sets of indices and prints the common and unique indices.

    Parameters:
    - set1: The first set of indices.
    - set2: The second set of indices.
    - set1_name: A string representing the name of the first set (for display purposes).
    - set2_name: A string representing the name of the second set (for display purposes).

    Returns:
    A dictionary containing:
    - 'common_indices': Indices present in both sets.
    - 'unique_to_set1': Indices unique to the first set.
    - 'unique_to_set2': Indices unique to the second set.
    """
    # Find indices that appear in both sets (common indices)
    common_indices = set1 & set2  # Intersection of the two sets

    # Find indices unique to each set
    unique_to_set1 = set1 - set2   # Indices in set1 but not in set2
    unique_to_set2 = set2 - set1   # Indices in set2 but not in set1

    # Output the results
    print(f"Indices present in both {set1_name} and {set2_name}:")
    print(sorted(common_indices))

    print(f"\nIndices unique to {set1_name}:")
    print(sorted(unique_to_set1))

    print(f"\nIndices unique to {set2_name}:")
    print(sorted(unique_to_set2))

    # Return the results in a dictionary
    return {
        'common_indices': common_indices,
        'unique_to_set1': unique_to_set1,
        'unique_to_set2': unique_to_set2
    }

In [23]:
compare_termination_initiation = compare_indices(PE6a_AF3_15A_set, PE6a_substrate_bound_AF3_15A_set, 'PE6c_AF3_15A', 'PE6c_substrate_bound_AF3_15A')

len(PE6a_AF3_15A_set)

Indices present in both PE6c_AF3_15A and PE6c_substrate_bound_AF3_15A:
[55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 136, 137, 138, 139, 140, 141, 142, 143, 171, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 213, 214, 215, 216, 247, 250, 263, 264, 265, 266, 273, 274, 275, 279, 282, 283, 286, 287, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 345, 349, 352, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 391, 394, 395, 397, 398, 399, 401, 402, 404, 405, 406, 407]

Indices unique to PE6c_AF3_15A:
[37, 39, 47, 54, 66, 144, 170, 176, 300, 

202

In [24]:
compare_termination_elongation = compare_indices(PE6a_AF3_18A_set, PE6a_substrate_bound_AF3_18A_set, 'PE6c_AF3_18A', 'PE6c_substrate_bound_AF3_18A')

len(PE6a_AF3_18A_set)

Indices present in both PE6c_AF3_18A and PE6c_substrate_bound_AF3_18A:
[19, 37, 38, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 167, 170, 171, 172, 173, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 211, 212, 213, 214, 215, 216, 217, 218, 244, 245, 246, 247, 248, 249, 250, 251, 262, 263, 264, 265, 266, 267, 272, 273, 274, 275, 276, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 293, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 344, 345, 346,

269

In [27]:
compare_termination_AF3 = compare_indices(PE6a_AF3_20A_set, PE6a_substrate_bound_AF3_20A_set, 'PE6c_AF3_20A', 'PE6c_substrate_bound_AF3_20A')


len(PE6a_AF3_20A_set)

Indices present in both PE6c_AF3_20A and PE6c_substrate_bound_AF3_20A:
[19, 20, 21, 22, 36, 37, 38, 39, 44, 47, 48, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 149, 152, 153, 156, 157, 167, 168, 169, 170, 171, 172, 173, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 211, 212, 213, 214, 215, 216, 217, 218, 239, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 261, 262, 263, 264, 265, 266, 267, 268, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 293, 294, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314,

304

## Generate final fixed residues by distance from combination of AF3 substrate bound structure and AF3 structure 

In [28]:
import re

# function to read the text file 
def read_file(file_path):
    with open(file_path, 'r') as infile:
        data = infile.read()
        # Split on commas and newlines, strip whitespace, and filter out empty strings
        data_list = [item.strip() for item in re.split(r',\s*|\n', data) if item.strip()]
        # Convert the list items to integers
        data_list = [int(item) for item in data_list]
    return data_list

# read the text file
PE6a_AF3_15A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6a/PE6a_AF3_15A_residue_indices.txt')
PE6a_substrate_bound_AF3_15A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6a/PE6a_substrate_bound_AF3_15A_residue_indices.txt')
PE6a_AF3_18A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6a/PE6a_AF3_18A_residue_indices.txt')
PE6a_substrate_bound_AF3_18A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6a/PE6a_substrate_bound_AF3_18A_residue_indices.txt')
PE6a_AF3_20A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6a/PE6a_AF3_20A_residue_indices.txt')
PE6a_substrate_bound_AF3_20A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6a/PE6a_substrate_bound_AF3_20A_residue_indices.txt')

In [29]:
# Convert the lists to sets for efficient comparison
PE6a_AF3_15A_set = set(PE6a_AF3_15A)
PE6a_substrate_bound_AF3_15A_set = set(PE6a_substrate_bound_AF3_15A)
PE6a_AF3_18A_set = set(PE6a_AF3_18A)
PE6a_substrate_bound_AF3_18A_set = set(PE6a_substrate_bound_AF3_18A)
PE6a_AF3_20A_set = set(PE6a_AF3_20A)
PE6a_substrate_bound_AF3_20A_set = set(PE6a_substrate_bound_AF3_20A)


In [30]:
def fixed_residues_distance(AF3, AF3_substrate_bound, output_file):
    
    fixed_residues = AF3_substrate_bound 

    # Find indices unique to each set
    unique_to_AF3= AF3 - AF3_substrate_bound   # Indices in AF3 but not in experiment

    for index in unique_to_AF3:
            fixed_residues.add(index)

    with open(output_file, 'w') as file:
        for residue in sorted(fixed_residues):  # Sort for readability
            file.write(f"{residue}\n")

    return fixed_residues

In [31]:
PE6a_fixed_residues_15A_output_file = '/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6a/PE6a_fixed_residues_15A.txt'
PE6a_fixed_residues_15A = fixed_residues_distance(PE6a_AF3_15A_set, PE6a_substrate_bound_AF3_15A_set, PE6a_fixed_residues_15A_output_file)
PE6a_fixed_residues_15A

{37,
 39,
 47,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 98,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 170,
 171,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,
 185,
 186,
 187,
 188,
 189,
 190,
 211,
 212,
 213,
 214,
 215,
 216,
 217,
 218,
 244,
 245,
 246,
 247,
 248,
 249,
 250,
 251,
 252,
 261,
 262,
 263,
 264,
 265,
 266,
 267,
 268,
 273,
 274,
 275,
 276,
 278,
 279,
 280,
 282,
 283,
 284,
 286,
 287,
 290,
 300,
 301,
 302,
 303,
 304,
 305,
 306,
 307,
 308,
 309,
 310,
 311,
 312,
 313,
 314,
 315,
 316,
 317,
 318,
 319,
 320,
 321,
 322,
 323,
 324,
 325,
 326,
 327,
 328,
 329,
 330,
 331,
 332,
 333,
 334,
 335,
 336,
 337,
 33

In [32]:
PE6a_fixed_residues_18A_output_file = '/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6a/PE6a_fixed_residues_18A.txt'
PE6a_fixed_residues_18A = fixed_residues_distance(PE6a_AF3_18A_set, PE6a_substrate_bound_AF3_18A_set, PE6a_fixed_residues_18A_output_file)
PE6a_fixed_residues_18A

{19,
 21,
 36,
 37,
 38,
 39,
 43,
 44,
 46,
 47,
 48,
 50,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 149,
 152,
 167,
 168,
 170,
 171,
 172,
 173,
 174,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,
 185,
 186,
 187,
 188,
 189,
 190,
 191,
 192,
 210,
 211,
 212,
 213,
 214,
 215,
 216,
 217,
 218,
 219,
 220,
 235,
 239,
 244,
 245,
 246,
 247,
 248,
 249,
 250,
 251,
 252,
 253,
 254,
 260,
 261,
 262,
 263,
 264,
 265,
 266,
 267,
 268,
 269,
 272,
 273,
 274,
 275,
 276,
 277,
 278,
 279,
 280,
 281,
 282,
 283,
 284,
 285,
 286

In [33]:
PE6a_fixed_residues_20A_output_file = '/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6a/PE6a_fixed_residues_20A.txt'
PE6a_fixed_residues_20A = fixed_residues_distance(PE6a_AF3_20A_set, PE6a_substrate_bound_AF3_20A_set, PE6a_fixed_residues_20A_output_file)
PE6a_fixed_residues_20A

{18,
 19,
 20,
 21,
 22,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 148,
 149,
 152,
 153,
 156,
 157,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,
 185,
 186,
 187,
 188,
 189,
 190,
 191,
 192,
 193,
 196,
 209,
 210,
 211,
 212,
 213,
 214,
 215,
 216,
 217,
 218,
 219,
 220,
 232,
 235,
 236,
 238,
 239,
 240,
 242,
 243,
 244,
 245,
 246,
 247,
 248,
 249